In [1]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

cwd = os.getcwd()
if cwd.endswith('notebook'):
    os.chdir('..')
    cwd = os.getcwd()

In [2]:
sns.set_palette('colorblind')
sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.8)
plt.rcParams['font.family'] = 'Helvetica'

palette = sns.color_palette().as_hex()

data_folder = Path('./data')
assert data_folder.is_dir()

figures_folder = Path('./figures')
assert figures_folder.is_dir()

In [3]:
gtdb_metadata = pd.read_csv(data_folder / 'gtdb_metadata.csv', index_col='ncbi_accession')
archaeal_accessions = set(gtdb_metadata[gtdb_metadata['domain'] == 'Archaea'].index)

In [41]:
pgh_df = pd.read_csv(data_folder / 'pgh_proteins.csv')
archaeal_pgh = pgh_df[pgh_df['domain'] == 'Archaea'].set_index('gtdb_class')
archaeal_pgh.head()

,assembly_accession,domain,gtdb_phylum,gtdb_order,gtdb_family,gtdb_genus,gtdb_species,ncbi_organism_name,protein_id,pgh_architecture
gtdb_class,,,,,,,,,,
Aenigmatarchaeia,GCA_003663345.1,Archaea,Aenigmatarchaeota,PWEA01,B50-G16,B50-G16,B50-G16 sp003663345,Candidatus Aenigmarchaeota archaeon,RLJ01288.1,PG_binding_3+Glyco_hydro_108
Aenigmatarchaeia,GCA_018304545.1,Archaea,Aenigmatarchaeota,CG10238-14,CG10238-14,JAGVVZ01,JAGVVZ01 sp018304545,Candidatus Aenigmarchaeota archaeon,MBS3052716.1,LysM+Amidase_2
Aenigmatarchaeia,GCA_025058005.1,Archaea,Aenigmatarchaeota,CG10238-14,CG10238-14,JAHLMN01,JAHLMN01 sp025058005,Candidatus Aenigmarchaeota archaeon,MCS7135447.1,LysM+NLPC_P60
Aenigmatarchaeia,GCA_015661515.1,Archaea,Aenigmatarchaeota,Aenigmatarchaeales,SZUA-1535,SZUA-1535,SZUA-1535 sp015661515,Nanoarchaeota archaeon,HIQ49991.1,PG_binding_3+Glyco_hydro_108
Altiarchaeia,GCA_902384455.1,Archaea,Altiarchaeota,IMC4,SCGC-AAA252-I15,CABMCB01,CABMCB01 sp902384455,uncultured archaeon,VVB52780.1,PG_binding_3+Glyco_hydro_108


In [44]:
catalytic_only = pd.read_csv(data_folder / 'catalytic_only_archaeal_proteins.csv')
catalytic_only = catalytic_only[catalytic_only['hmm_query'] != 'Amidase'].reset_index(drop=True)
catalytic_only.head()

,assembly_accession,protein_id,hmm_accession,hmm_query,domain,gtdb_phylum,gtdb_class,gtdb_order,gtdb_family,gtdb_genus,gtdb_species,id
0,GCA_000220355.1,EGQ39878.1,PF17676.4,Peptidase_S66C,Archaea,Nanohaloarchaeota,Nanosalinia,Nanosalinales,Nanosalinaceae,Nanosalinarum,Nanosalinarum sp000220355,EGQ39878.1@GCA_000220355.1
1,GCA_000220355.1,EGQ39880.1,PF02016.18,Peptidase_S66,Archaea,Nanohaloarchaeota,Nanosalinia,Nanosalinales,Nanosalinaceae,Nanosalinarum,Nanosalinarum sp000220355,EGQ39880.1@GCA_000220355.1
2,GCA_000220355.1,EGQ39880.1,PF17676.4,Peptidase_S66C,Archaea,Nanohaloarchaeota,Nanosalinia,Nanosalinales,Nanosalinaceae,Nanosalinarum,Nanosalinarum sp000220355,EGQ39880.1@GCA_000220355.1
3,GCA_000220355.1,EGQ39881.1,PF02016.18,Peptidase_S66,Archaea,Nanohaloarchaeota,Nanosalinia,Nanosalinales,Nanosalinaceae,Nanosalinarum,Nanosalinarum sp000220355,EGQ39881.1@GCA_000220355.1
4,GCA_000220355.1,EGQ39881.1,PF17676.4,Peptidase_S66C,Archaea,Nanohaloarchaeota,Nanosalinia,Nanosalinales,Nanosalinaceae,Nanosalinarum,Nanosalinarum sp000220355,EGQ39881.1@GCA_000220355.1


In [35]:
pg_uptake_archaea = pd.read_csv(data_folder / 'pg_uptake' / 'pg_uptake_archaeal_hits.csv')

pg_uptake_archaea = pd.merge(
    pg_uptake_archaea,
    gtdb_metadata.reset_index()[
        ['ncbi_accession', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 'gtdb_genus', 'gtdb_species']
    ].rename(columns={
        'ncbi_accession': 'assembly_accession',
    }),
    on='assembly_accession',
    how='left',
)

pg_uptake_archaea['uniprot_id'] = pg_uptake_archaea['query_name'].apply(lambda v: v.split('|')[1])

pg_uptake_archaea = pg_uptake_archaea.sort_values(
    ['target_name', 'score_full'], ascending=[True, False]
).drop_duplicates('target_name')

pg_uptake_archaea.head()

,target_name,target_accession,query_name,query_accession,evalue_full,score_full,bias_full,evalue_best_dom,score_best_dom,bias_best_dom,...,rep,inc,assembly_accession,gtdb_phylum,gtdb_class,gtdb_order,gtdb_family,gtdb_genus,gtdb_species,uniprot_id
116709,AAR39147.1@GCA_000008085.1,-,sp|P77737|OPPF_ECOLI,-,2.200000e-18,79.3,1.1,5.500000e-05,35.2,0.0,...,2,2,GCA_000008085.1,Nanoarchaeota,Nanoarchaeia,Nanoarchaeales,Nanoarchaeaceae,Nanoarchaeum,Nanoarchaeum equitans,P77737
189225,AAR39266.1@GCA_000008085.1,-,sp|A9CKL2|YEJF_AGRFC,-,4.800000e-25,101.1,0.0,5.900000e-25,100.7,0.0,...,1,1,GCA_000008085.1,Nanoarchaeota,Nanoarchaeia,Nanoarchaeales,Nanoarchaeaceae,Nanoarchaeum,Nanoarchaeum equitans,A9CKL2
90636,ABK77013.1@GCA_000200715.1,-,sp|P77737|OPPF_ECOLI,-,5.400000e-61,219.4,0.1,6.900000e-61,219.0,0.1,...,1,1,GCA_000200715.1,Thermoproteota,Nitrososphaeria,Nitrososphaerales,Nitrosopumilaceae,Cenarchaeum,Cenarchaeum symbiosum,P77737
246251,ABK77177.1@GCA_000200715.1,-,sp|A9CKL2|YEJF_AGRFC,-,9.300000e-09,47.3,4.5,2.300000e-01,22.9,0.0,...,2,1,GCA_000200715.1,Thermoproteota,Nitrososphaeria,Nitrososphaerales,Nitrosopumilaceae,Cenarchaeum,Cenarchaeum symbiosum,A9CKL2
35380,ABK77211.1@GCA_000200715.1,-,sp|P76027|OPPD_ECOLI,-,1.600000e-18,79.8,0.0,2.100000e-18,79.5,0.0,...,1,1,GCA_000200715.1,Thermoproteota,Nitrososphaeria,Nitrososphaerales,Nitrosopumilaceae,Cenarchaeum,Cenarchaeum symbiosum,P76027


In [36]:
high_level_stats = pg_uptake_archaea[
    ['query_name', 'assembly_accession', 'gtdb_phylum']
].groupby(['query_name']).nunique().sort_values('assembly_accession', ascending=False)

high_level_stats['percent_genomes'] = (100 * high_level_stats['assembly_accession'] / len(archaeal_accessions)).round(1)

high_level_stats

,assembly_accession,gtdb_phylum,percent_genomes
query_name,,,
sp|A9CKL2|YEJF_AGRFC,3558,20,96.0
sp|P77737|OPPF_ECOLI,3547,21,95.7
sp|P76027|OPPD_ECOLI,3000,19,80.9
sp|P0AFH6|OPPC_ECOLI,1685,11,45.5
sp|P0AFH2|OPPB_ECOLI,1378,11,37.2
sp|A9CKL3|YEJB_AGRFC,1046,6,28.2
sp|P77348|MPPA_ECOLI,711,10,19.2
sp|P0AAE8|CADB_ECOLI,659,14,17.8
sp|P23843|OPPA_ECOLI,654,12,17.6


In [37]:
pg_uptake_archaea[
    pg_uptake_archaea['query_name'] == 'sp|P0AE16|AMPG_ECOLI'
][
    ['query_name', 'assembly_accession', 'gtdb_phylum']
].groupby(['gtdb_phylum', 'query_name']).nunique().sort_values('assembly_accession', ascending=False)

,,assembly_accession
gtdb_phylum,query_name,
Thermoplasmatota,sp|P0AE16|AMPG_ECOLI,68
Asgardarchaeota,sp|P0AE16|AMPG_ECOLI,5
Halobacteriota,sp|P0AE16|AMPG_ECOLI,1


### Pathways

In [38]:
pathways = {
    'OppBCDF-MppA': ['P23843', 'P0AFH2', 'P0AFH6', 'P76027', 'P77737', 'P77348'],
    'AmpG': ['P0AE16'],
    'YejBEF-YepA': ['A9CKL4', 'A9CKL3', 'Q7D203', 'A9CKL2', 'A9CIN5'],
    'CadB': ['P0AAE8'],
    'NagE': ['P09323'],
    'MurP': ['P77272'],
    'MurT': ['G8UQH8'],
}

In [39]:
pathway_to_accessions = {}
for pathway, uniprot_ids in pathways.items():
    df = pg_uptake_archaea[pg_uptake_archaea['uniprot_id'].isin(uniprot_ids)]
    g = df[['assembly_accession', 'uniprot_id']].groupby('assembly_accession').nunique()
    accessions = sorted(set(g[g['uniprot_id'] == len(uniprot_ids)].index))
    pathway_to_accessions[pathway] = accessions

In [66]:
accessions_pgh = set(archaeal_pgh['assembly_accession'].unique())

all_accessions = set()
for pathway in pathways.keys():
    accessions = set(pathway_to_accessions[pathway])
    n = len(accessions)
    p = 100 * n / len(archaeal_accessions)

    n_with_pgh = len(accessions & accessions_pgh)
    n_no_pgh = len(accessions_pgh - accessions)

    no_pathway = set(archaeal_accessions) - accessions

    all_accessions |= accessions

    print(f'{pathway}:')
    print(f'\tTotal = {n:,} of {len(archaeal_accessions):,} ({p:.1f}%)')
    print(f'\tPGH & {pathway} = {n_with_pgh:,} of {len(accessions):,} ({100 * n_with_pgh / len(accessions):.1f}%)')
    print(f'\tPGH no {pathway} = {n_no_pgh:,} of {len(no_pathway):,} ({100 * n_no_pgh / len(no_pathway):.1f}%)')

OppBCDF-MppA:
	Total = 338 of 3,706 (9.1%)
	PGH & OppBCDF-MppA = 40 of 338 (11.8%)
	PGH no OppBCDF-MppA = 135 of 3,368 (4.0%)
AmpG:
	Total = 74 of 3,706 (2.0%)
	PGH & AmpG = 1 of 74 (1.4%)
	PGH no AmpG = 174 of 3,632 (4.8%)
YejBEF-YepA:
	Total = 2 of 3,706 (0.1%)
	PGH & YejBEF-YepA = 0 of 2 (0.0%)
	PGH no YejBEF-YepA = 175 of 3,704 (4.7%)
CadB:
	Total = 659 of 3,706 (17.8%)
	PGH & CadB = 38 of 659 (5.8%)
	PGH no CadB = 137 of 3,047 (4.5%)
NagE:
	Total = 2 of 3,706 (0.1%)
	PGH & NagE = 0 of 2 (0.0%)
	PGH no NagE = 175 of 3,704 (4.7%)
MurP:
	Total = 2 of 3,706 (0.1%)
	PGH & MurP = 0 of 2 (0.0%)
	PGH no MurP = 175 of 3,704 (4.7%)
MurT:
	Total = 14 of 3,706 (0.4%)
	PGH & MurT = 0 of 14 (0.0%)
	PGH no MurT = 175 of 3,692 (4.7%)


In [67]:
n = len(all_accessions)
p = 100 * n / len(archaeal_accessions)

n_with = len(all_accessions & accessions_pgh)
n_no = len(accessions_pgh - all_accessions)

no_pathway = set(archaeal_accessions) - all_accessions

print(f'Overall:')
print(f'\tTotal = {n:,} of {len(archaeal_accessions):,} ({p:.1f}%)')
print(f'\tCatalytic only & uptake pathway = {n_with:,} of {len(all_accessions):,} ({100 * n_with / len(all_accessions):.1f}%)')
print(f'\tCatalytic only & no uptake pathway = {n_no:,} of {len(no_pathway):,} ({100 * n_no / len(no_pathway):.1f}%)')

Overall:
	Total = 963 of 3,706 (26.0%)
	Catalytic only & uptake pathway = 57 of 963 (5.9%)
	Catalytic only & no uptake pathway = 118 of 2,743 (4.3%)


In [64]:
catalytic_only_acc = set(catalytic_only['assembly_accession'].unique())

all_accessions = set()
for pathway in pathways.keys():
    accessions = set(pathway_to_accessions[pathway])
    n = len(accessions)
    p = 100 * n / len(archaeal_accessions)

    n_with = len(accessions & catalytic_only_acc)
    n_no = len(catalytic_only_acc - accessions)

    no_pathway = set(archaeal_accessions) - accessions

    all_accessions |= accessions

    print(f'{pathway}:')
    print(f'\tTotal = {n:,} of {len(archaeal_accessions):,} ({p:.1f}%)')
    print(f'\tCatalytic only & {pathway} = {n_with:,} of {len(accessions):,} ({100 * n_with / len(accessions):.1f}%)')
    print(f'\tCatalytic only & no {pathway} = {n_no:,} of {len(no_pathway):,} ({100 * n_no / len(no_pathway):.1f}%)')

OppBCDF-MppA:
	Total = 338 of 3,706 (9.1%)
	Catalytic only & OppBCDF-MppA = 224 of 338 (66.3%)
	Catalytic only & no OppBCDF-MppA = 1,808 of 3,368 (53.7%)
AmpG:
	Total = 74 of 3,706 (2.0%)
	Catalytic only & AmpG = 72 of 74 (97.3%)
	Catalytic only & no AmpG = 1,960 of 3,632 (54.0%)
YejBEF-YepA:
	Total = 2 of 3,706 (0.1%)
	Catalytic only & YejBEF-YepA = 1 of 2 (50.0%)
	Catalytic only & no YejBEF-YepA = 2,031 of 3,704 (54.8%)
CadB:
	Total = 659 of 3,706 (17.8%)
	Catalytic only & CadB = 361 of 659 (54.8%)
	Catalytic only & no CadB = 1,671 of 3,047 (54.8%)
NagE:
	Total = 2 of 3,706 (0.1%)
	Catalytic only & NagE = 1 of 2 (50.0%)
	Catalytic only & no NagE = 2,031 of 3,704 (54.8%)
MurP:
	Total = 2 of 3,706 (0.1%)
	Catalytic only & MurP = 2 of 2 (100.0%)
	Catalytic only & no MurP = 2,030 of 3,704 (54.8%)
MurT:
	Total = 14 of 3,706 (0.4%)
	Catalytic only & MurT = 9 of 14 (64.3%)
	Catalytic only & no MurT = 2,023 of 3,692 (54.8%)


In [65]:
n = len(all_accessions)
p = 100 * n / len(archaeal_accessions)

n_with = len(all_accessions & catalytic_only_acc)
n_no = len(catalytic_only_acc - all_accessions)

no_pathway = set(archaeal_accessions) - all_accessions

print(f'Overall:')
print(f'\tTotal = {n:,} of {len(archaeal_accessions):,} ({p:.1f}%)')
print(f'\tCatalytic only & uptake pathway = {n_with:,} of {len(all_accessions):,} ({100 * n_with / len(all_accessions):.1f}%)')
print(f'\tCatalytic only & no uptake pathway = {n_no:,} of {len(no_pathway):,} ({100 * n_no / len(no_pathway):.1f}%)')

Overall:
	Total = 963 of 3,706 (26.0%)
	Catalytic only & uptake pathway = 577 of 963 (59.9%)
	Catalytic only & no uptake pathway = 1,455 of 2,743 (53.0%)
